In [1]:
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from dotenv import load_dotenv

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-3.7-flash")

In [2]:
class LinkedInState(TypedDict):
    topic: str
    post: str
    approved: bool

In [3]:
def generate_post(state: LinkedInState):
    prompt = f"""
    Write an engaging LinkedIn post about {state['topic']}.
    Keep it professional, natural, and suitable for LinkedIn.
    """

    result = llm.invoke(prompt).content

    return {"post": result}

In [4]:
def human_review(state: LinkedInState):
    response = interrupt({
        "message": "Review this LinkedIn post. Approve or reject it.",
        "post": state["post"],
        "options": ["approve", "reject"]
    })

    return {
        "approved": response.lower() == "approve"
    }

In [5]:
graph = StateGraph(LinkedInState)

graph.add_node("generate_post", generate_post)
graph.add_node("human_review", human_review)

graph.add_edge(START, "generate_post")
graph.add_edge("generate_post", "human_review")
graph.add_edge("human_review", END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [7]:
config = {
    "configurable": {
        "thread_id": "linkedin-post-1"
    }
}

result = workflow.invoke(
    {
        "topic": "How AI is changing backend development",
        "post": "",
        "approved": False
    },
    config=config
)

print(result["__interrupt__"])

[Interrupt(value={'message': 'Review this LinkedIn post. Approve or reject it.', 'post': [{'type': 'text', 'text': "Frontend usually gets the flashiest AI demos, but the quietest—and most profound—shift is happening in backend development.\n\nFor a long time, backend work was defined by a balance: 30% deep architectural thinking, and 70% repetitive plumbing (writing boilerplate CRUD, mapping schemas, configuring pipelines, and debugging obscure logs).\n\nAI is flipping that ratio.\n\nHere is what that actually looks like in practice right now:\n\n1. **Boilerplate is becoming obsolete**\nGenerating repetitive DTOs, standard endpoints, and database migrations used to eat up hours. Now, an engineer can describe the business domain and generate clean, standardized boilerplate in seconds. \n\n2. **The test coverage gap is shrinking**\nWriting unit and integration tests is critical, but often compromised under tight deadlines. AI has become an incredible pair programmer for mocking data, gen